# CBFpy Examples

This notebook demonstrates how to use the `CBF` and `CBFConfig` classes from the `cbfpy` package.

In [1]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from cbfpy import CBF, CBFConfig

In [3]:
import jax.numpy as jnp
from cbfpy import CBF, CBFConfig

# Create a config class for your problem inheriting from the CBFConfig class
class MyCBFConfig(CBFConfig):
    def __init__(self):
        super().__init__(
            # Define the state and control dimensions
            n = 2, # [x, x_dot]
            m = 1, # [F_x]
            # Define control limits (if desired)
            u_min = None,
            u_max = None,
        )

    # Define the control-affine dynamics functions `f` and `g` for your system
    def f(self, z):
        A = jnp.array([[0.0, 1.0], [0.0, 0.0]])
        return A @ z

    def g(self, z):
        mass = 1.0
        B = jnp.array([[0.0], [1.0 / mass]])
        return B

    # Define the barrier function `h`
    # The *relative degree* of this system is 2, so, we'll use the h_2 method
    def h_2(self, z):
        x_min = 1.0
        x = z[0]
        return jnp.array([x - x_min])

In [4]:
config = MyCBFConfig()
cbf = CBF.from_config(config)

# Pseudocode
while True:
    z = get_state()
    z_des = get_desired_state()
    u_nom = nominal_controller(z, z_des)
    u = cbf.safety_filter(z, u_nom)
    apply_control(u)
    step()

NameError: name 'get_state' is not defined

## Example 1: Velocity Limit (Relative Degree 1)
System dynamics: $\dot{v} = -0.1v + u$
Safety constraint: $v \le 10 \implies h_1(v) = 10 - v \ge 0$

In [2]:
class VelocityLimitConfig(CBFConfig):
    def __init__(self):
        super().__init__(n=1, m=1)
        
    def f(self, z):
        return jnp.array([-0.1 * z[0]])
        
    def g(self, z):
        return jnp.array([[1.0]])
        
    def h_1(self, z):
        return jnp.array([10.0 - z[0]])
        
config = VelocityLimitConfig()
cbf = CBF.from_config(config)


In [3]:
import ipywidgets as widgets
from IPython.display import display

def simulate_velocity_limit(limit=10.0):
    # Re-define config and cbf to use the dynamic limit
    class DynamicVelocityConfig(CBFConfig):
        def __init__(self):
            super().__init__(n=1, m=1)
        def f(self, z):
            return jnp.array([-0.1 * z[0]])
        def g(self, z):
            return jnp.array([[1.0]])
        def h_1(self, z):
            return jnp.array([limit - z[0]])
            
    cbf = CBF.from_config(DynamicVelocityConfig())
    
    dt = 0.05
    T = 15.0
    steps = int(T / dt)
    v = 0.0
    v_history = []
    u_history = []
    
    for _ in range(steps):
        z = jnp.array([v])
        u_nom = jnp.array([5.0]) # Always try to accelerate at 5.0
        
        # Get safe control
        u_safe = cbf.safety_filter(z, u_nom)
        
        v_history.append(v)
        u_history.append(float(u_safe[0]))
        
        # update dynamics (Euler integration)
        dv = -0.1 * v + float(u_safe[0])
        v += dv * dt
        
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(jnp.arange(steps)*dt, v_history)
    plt.axhline(limit, color='r', linestyle='--', label='Limit')
    plt.title('Velocity vs Time')
    plt.xlabel('Time (s)')
    plt.ylabel('Velocity')
    plt.ylim(0, 25)
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(jnp.arange(steps)*dt, u_history)
    plt.title('Control Input vs Time')
    plt.xlabel('Time (s)')
    plt.ylabel('Control u')
    plt.ylim(-2, 6)
    plt.tight_layout()
    plt.show()

# Create interactive slider
widgets.interact(simulate_velocity_limit, limit=widgets.FloatSlider(value=10.0, min=1.0, max=20.0, step=1.0, description='Vel Limit:'))


interactive(children=(FloatSlider(value=10.0, description='Vel Limit:', max=20.0, min=1.0, step=1.0), Output()…

<function __main__.simulate_velocity_limit(limit=10.0)>

## Example 2: Car Follower (Adaptive Cruise Control)

**System dynamics (relative degree 1):**
- Ego velocity: $\dot{v} = u$
- Distance to lead car: $\dot{D} = v_L - v$ (where $v_L$ is the constant velocity of the lead car)

**Safety constraint:** Keep a safe distance $D \ge T_h v + D_{min}$
- $h_1(z) = D - T_h v - D_{min} \ge 0$
where $T_h$ is the time headway and $D_{min}$ is the minimum distance.

In [1]:
from cbfpy.config.clf_cbf_config import CLFCBFConfig

In [2]:
class ACCConfig(CLFCBFConfig):
    """Configuration for the Adaptive Cruise Control CLF-CBF demo"""

    def __init__(self):
        self.gravity = 9.81
        self.mass = 1650.0
        self.drag_coeffs = (0.1, 5.0, 0.25)  # Drag coeffs
        self.v_des = 24.0  # Desired velocity
        self.T = 1.8  # Lookahead time
        self.cd = 0.3  # Coefficient of maximum deceleration
        self.ca = 0.3  # Coefficient of maximum acceleration
        u_min = -self.cd * self.mass * self.gravity  # Min. control input (max braking)
        u_max = self.ca * self.mass * self.gravity  # Max. control input (max throttle)
        super().__init__(
            n=3,
            m=1,
            u_min=u_min,
            u_max=u_max,
            # Note: Relaxing the CLF-CBF QP is tricky because there is an additional relaxation
            # parameter already, balancing the CLF and CBF constraints.
            relax_qp=False,
            # If indeed relaxing, ensure that the CBF relaxation >> the CLF relaxation
            clf_relaxation_penalty=10.0,
            cbf_relaxation_penalty=1e5,
            control_relaxation_penalty=1e6,
        )

    def drag_force(self, v: float) -> float:
        """Compute the drag force on the follower car using a simple polynomial model

        Args:
            v (float): Velocity of the follower vehicle, in m/s

        Returns:
            float: Drag force, in Newtons
        """
        return (
            self.drag_coeffs[0] + self.drag_coeffs[1] * v + self.drag_coeffs[2] * v**2
        )

    def f(self, z: ArrayLike) -> Array:
        v_f, v_l, D = z
        # Note: We assume that the leader vehicle is at constant velocity here
        return jnp.array([-self.drag_force(v_f) / self.mass, 0.0, v_l - v_f])

    def g(self, z: ArrayLike) -> Array:
        return jnp.array([(1 / self.mass), 0.0, 0.0]).reshape(-1, 1)

    def h_1(self, z: ArrayLike) -> Array:
        v_f, v_l, D = z
        return jnp.array(
            [D - self.T * v_f - 0.5 * (v_l - v_f) ** 2 / (self.cd * self.gravity)]
        )

    def V_1(self, z: ArrayLike, z_des: ArrayLike) -> float:
        # CLF: Squared error between the follower velocity and the desired velocity
        return jnp.array([(z[0] - self.v_des) ** 2])

    def H(self, z: ArrayLike) -> Array:
        return jnp.eye(self.m) * (2 / self.mass**2)

    def F(self, z: ArrayLike) -> Array:
        return jnp.array([-2 * self.drag_force(z[0]) / self.mass**2])

# Create CBF
acc_config = ACCConfig(v_L=15.0)
acc_cbf = CBF.from_config(acc_config)

# Simulate
dt = 0.05
T = 30.0
steps = int(T / dt)

# Initial state: fast ego car, close to lead car
v = 20.0
D = 50.0

history_v = []
history_D = []
history_u = []
history_h = []

for _ in range(steps):
    z = jnp.array([v, D])
    # nominal control: try to reach 25 m/s
    u_nom = jnp.array([1.0 * (25.0 - v)]) 
    
    u_safe = acc_cbf.safety_filter(z, u_nom)
    
    history_v.append(v)
    history_D.append(D)
    history_u.append(float(u_safe[0]))
    history_h.append(float(acc_config.h_1(z)[0]))
    
    # update dynamics
    dv = float(u_safe[0])
    dD = 15.0 - v
    v += dv * dt
    D += dD * dt

# Plotting
time = jnp.arange(steps) * dt
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(time, history_v, label='Ego Vel')
plt.axhline(15.0, color='r', linestyle='--', label='Lead Vel')
plt.title('Velocity vs Time')
plt.xlabel('Time (s)')
plt.ylabel('Velocity (m/s)')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(time, history_D, label='Distance')
safe_D = 1.5 * jnp.array(history_v) + 5.0
plt.plot(time, safe_D, 'r--', label='Min Safe Dist')
plt.title('Distance to Lead Car')
plt.xlabel('Time (s)')
plt.ylabel('Distance (m)')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(time, history_h, label='h(x)')
plt.axhline(0.0, color='r', linestyle='--')
plt.title('CBF Value h(x)')
plt.xlabel('Time (s)')
plt.ylabel('h(x)')
plt.legend()

plt.tight_layout()
plt.show()


TypeError: ACCConfig.__init__() got an unexpected keyword argument 'v_L'